In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load processed data
df = pd.read_csv("../data/processed/AAPL_processed.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

# Features
df["lag_vol"] = df["log_return"].rolling(21).std() * np.sqrt(252) * 100  # rolling vol
df["lag1"] = df["log_return"].shift(1)
df["lag2"] = df["log_return"].shift(2)
df["lag3"] = df["log_return"].shift(3)
df.dropna(inplace=True)

X = df[["lag_vol", "lag1", "lag2", "lag3"]]
y = np.sqrt(df["squared_return"]) * 100  # realized volatility target

# Train/test split (time-based)
split = int(0.8 * len(X))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# XGBoost
xgb = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

# Evaluation
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{name} → RMSE={rmse:.4f}, MAE={mae:.4f}")

evaluate("Random Forest", y_test, rf_preds)
evaluate("XGBoost", y_test, xgb_preds)

# Save results for dashboard
out = pd.DataFrame({"RF": rf_preds, "XGB": xgb_preds}, index=X_test.index)
out.to_csv("../data/processed/ml_forecasts.csv", index_label="Date")


Random Forest → RMSE=1.2279, MAE=0.8747
XGBoost → RMSE=1.2340, MAE=0.8494
